# 95 — Pipeline: 89 SELECTED → 94 M6（一括）

89（mass explore）が出し続けている SELECTED を、
**90 advance → 91 M3 → 92 M4/M5 → 93 obligations → 94 M6** まで一気に進める。

- 89 と **並行可**（このノートは消費側；新規 SELECTED は次ラウンドで拾う）
- `MAX_ROUNDS` を上げると、進捗がある限り繰り返し吸い上げる
- production paperspace M6（81）は起動しない
- 常に `NOT_CERTIFIED` / `SCREENING_ONLY`（M6 が CERTIFIED でも continuum 主張禁止）
- GPU が必要な段（91/92）は CUDA 必須


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'pipeline_to_m6.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/pipeline_to_m6.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M3_ALLOW_CODE_DRIFT', '1')
os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 95 requires CUDA for M3/M4 stages.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.pipeline_to_m6 import run_pipeline_to_m6

validate_persistent_root()
# Knobs — raise MAX_* to drain more of the 89 backlog per round:
MAX_ROUNDS = 20                 # re-scan while 89 keeps adding SELECTED
MAX_ADVANCE = None              # None = all discovered SELECTED
MAX_M3_SESSIONS = 16
MAX_PRE_M6_PACKAGES = 16
MAX_STAGE_SESSIONS = 6          # M4 resume loops per package (notebook 92)
MAX_OBLIGATION_PACKAGES = 16
MAX_M6_PACKAGES = 16
MAX_QUEUE = 2000
ONLY_CAMPAIGN = None            # or pin one M7-... campaign id

print(json.dumps({
    'MAX_ROUNDS': MAX_ROUNDS,
    'MAX_ADVANCE': MAX_ADVANCE,
    'MAX_M3_SESSIONS': MAX_M3_SESSIONS,
    'MAX_PRE_M6_PACKAGES': MAX_PRE_M6_PACKAGES,
    'MAX_STAGE_SESSIONS': MAX_STAGE_SESSIONS,
    'MAX_OBLIGATION_PACKAGES': MAX_OBLIGATION_PACKAGES,
    'MAX_M6_PACKAGES': MAX_M6_PACKAGES,
    'MAX_QUEUE': MAX_QUEUE,
    'ONLY_CAMPAIGN': ONLY_CAMPAIGN,
}, indent=2))


In [ ]:
SESSION = run_pipeline_to_m6(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    max_rounds=MAX_ROUNDS,
    max_advance=MAX_ADVANCE,
    max_m3_sessions=MAX_M3_SESSIONS,
    max_pre_m6_packages=MAX_PRE_M6_PACKAGES,
    max_stage_sessions=MAX_STAGE_SESSIONS,
    max_obligation_packages=MAX_OBLIGATION_PACKAGES,
    max_m6_packages=MAX_M6_PACKAGES,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'rounds_run': SESSION.get('rounds_run'),
    'totals': SESSION.get('totals'),
    'rounds': [
        {
            'round': r.get('round'),
            'progress': r.get('progress'),
            'stages': {
                k: {
                    kk: vv for kk, vv in (v or {}).items()
                    if kk != 'results' and kk != 'errors'
                }
                for k, v in (r.get('stages') or {}).items()
            },
        }
        for r in (SESSION.get('rounds') or [])
    ],
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_pipeline_to_m6' / 'LATEST_PIPELINE_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
